# Antenna / Cell Tower Detection with YOLO11

A complete end-to-end pipeline for training a custom YOLO11 object detector to identify antenna / cell towers in imagery.

---


## 1. Environment Setup

Check GPU and install required libraries.


In [ ]:
!nvidia-smi

import torch
print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    device = 0
else:
    print("WARNING: Running on CPU (very slow for training!)")
    device = "cpu"


In [ ]:
!pip install -U ultralytics -q

import ultralytics
from ultralytics import YOLO

print("\nUltralytics version:", ultralytics.__version__)
print("YOLO11 library loaded successfully.")


## 2. Upload Dataset (ZIP)

Upload your YOLO-format dataset ZIP (e.g. exported from Roboflow).

Expected structure:
```
dataset/
├── data.yaml
├── train/images  train/labels
├── valid/images  valid/labels
└── test/images   test/labels   (optional)
```


In [ ]:
from google.colab import files
import os
import glob
import zipfile

print("Upload your dataset ZIP file.")
uploaded = files.upload()

zip_file = None
for filename in uploaded.keys():
    if filename.lower().endswith(".zip"):
        zip_file = filename
        break

if zip_file is None:
    raise FileNotFoundError("No ZIP file uploaded.")

print("\nUploaded:", zip_file)


In [ ]:
os.makedirs("/content/dataset", exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("/content/dataset")

print("Dataset extracted successfully.")


In [ ]:
yaml_files = glob.glob("/content/dataset/**/data.yaml", recursive=True)
if len(yaml_files) == 0:
    raise FileNotFoundError("data.yaml was not found inside the uploaded dataset.")

data_yaml_original = yaml_files[0]
dataset_root = os.path.dirname(data_yaml_original)

print("Original data.yaml:", data_yaml_original)
print("Dataset root:", dataset_root)
print("\nContents:")
for item in sorted(os.listdir(dataset_root)):
    print(" -", item)


In [ ]:
required_folders = [
    "train/images", "train/labels",
    "valid/images", "valid/labels",
    "test/images",  "test/labels",
]

print("Checking dataset structure...\n")
for folder in required_folders:
    fp = os.path.join(dataset_root, folder)
    if os.path.exists(fp):
        n = len(os.listdir(fp))
        print(f"FOUND    : {folder:18s} ({n} items)")
    else:
        print(f"NOT FOUND: {folder}   (optional if test)")


## 3. Prepare data.yaml with absolute paths


In [ ]:
import yaml
from pathlib import Path

with open(data_yaml_original, "r") as f:
    data = yaml.safe_load(f)

data["path"] = dataset_root
data["train"] = str(Path(dataset_root, "train", "images"))
data["val"]   = str(Path(dataset_root, "valid", "images"))

test_images_path = Path(dataset_root, "test", "images")
if test_images_path.exists():
    data["test"] = str(test_images_path)

data_yaml = "/content/data.yaml"
with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Corrected data.yaml saved to:", data_yaml)
print("\n========================================")
print("FINAL data.yaml")
print("========================================\n")
with open(data_yaml, "r") as f:
    print(f.read())


In [ ]:
names = data["names"]

print("========================================")
print("DATASET CLASSES")
print("========================================")
if isinstance(names, dict):
    sorted_keys = sorted(names.keys(), key=lambda x: int(x))
    class_names = [names[k] for k in sorted_keys]
    for k in sorted_keys:
        print(f"{k}: {names[k]}")
else:
    class_names = list(names)
    for idx, name in enumerate(class_names):
        print(f"{idx}: {name}")

print("\nNumber of classes:", len(class_names))


In [ ]:
def count_images(split):
    p = Path(dataset_root, split, "images")
    return len(list(p.glob("*"))) if p.exists() else 0

print("========================================")
print("DATASET SIZE")
print("========================================")
print("Training images  :", count_images("train"))
print("Validation images:", count_images("valid"))
print("Test images      :", count_images("test"))


## 4. Load & Train YOLO11 Nano


In [ ]:
model = YOLO("yolo11n.pt")
print("YOLO11 Nano model loaded successfully.")


In [ ]:
print("========================================")
print("STARTING YOLO11 TRAINING")
print("========================================\n")

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    patience=20,
    save=True,
    save_period=5,
    seed=42,
    project="/content/tower_detection",
    name="yolo11_tower_detector",
)


In [ ]:
training_folder = "/content/tower_detection/yolo11_tower_detector"
best_model_path = training_folder + "/weights/best.pt"
last_model_path = training_folder + "/weights/last.pt"

print("========================================")
print("TRAINING COMPLETED")
print("========================================")
print("Training folder :", training_folder)
print("Best model      :", best_model_path)
print("Last model      :", last_model_path)


## 5. Validation (val split)


In [ ]:
best_model = YOLO(best_model_path)
print("Best YOLO11 model loaded.")

print("\n========================================")
print("VALIDATION")
print("========================================\n")

val_metrics = best_model.val(
    data=data_yaml,
    split="val",
    imgsz=640,
    batch=16,
    device=device,
)


In [ ]:
print("========================================")
print("VALIDATION RESULTS")
print("========================================")
print("mAP50-95 :", val_metrics.box.map)
print("mAP50    :", val_metrics.box.map50)
print("mAP75    :", val_metrics.box.map75)

print("\n========================================")
print("PER-CLASS VALIDATION mAP50-95")
print("========================================")
for class_id, class_name in enumerate(class_names):
    class_map = val_metrics.box.maps[class_id]
    print(f"{class_id} - {class_name}: {class_map:.4f}")


## 6. Test split evaluation (if present)


In [ ]:
test_folder = os.path.join(dataset_root, "test", "images")

if os.path.exists(test_folder):
    print("========================================")
    print("TEST DATASET EVALUATION")
    print("========================================\n")
    test_metrics = best_model.val(
        data=data_yaml,
        split="test",
        imgsz=640,
        batch=16,
        device=device,
    )
    print("========================================")
    print("TEST RESULTS")
    print("========================================")
    print("Test mAP50-95 :", test_metrics.box.map)
    print("Test mAP50    :", test_metrics.box.map50)
    print("Test mAP75    :", test_metrics.box.map75)
else:
    print("Test images folder not found. Skipping test evaluation.")


## 7. Run Inference on Test Images


In [ ]:
if os.path.exists(test_folder):
    print("========================================")
    print("RUNNING DETECTION ON TEST IMAGES")
    print("========================================\n")
    prediction_results = best_model.predict(
        source=test_folder,
        imgsz=640,
        conf=0.25,
        iou=0.45,
        device=device,
        save=True,
        project="/content/tower_detection",
        name="test_predictions",
    )
else:
    prediction_results = []


In [ ]:
from PIL import Image
from IPython.display import display

prediction_folder = "/content/tower_detection/test_predictions"
result_images = glob.glob(os.path.join(prediction_folder, "*"))

print("========================================")
print("DETECTION RESULTS")
print("========================================")
print("Number of result images:", len(result_images))

for image_path in result_images[:10]:
    try:
        print("\nImage:", os.path.basename(image_path))
        display(Image.open(image_path))
    except Exception:
        pass


In [ ]:
print("========================================")
print("PER-IMAGE OBJECT DETECTIONS")
print("========================================")

for result in prediction_results:
    path = result.path
    boxes = result.boxes
    print(f"\n{os.path.basename(path)}: {len(boxes)} detections")
    if len(boxes) > 0:
        for box in boxes:
            cls_id = int(box.cls[0])
            cls_name = class_names[cls_id] if cls_id < len(class_names) else f"cls_{cls_id}"
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            print(f"  - {cls_name:20s}  conf={conf:.2f}  box=({x1},{y1})-({x2},{y2})")


## 8. Download trained models


In [ ]:
from google.colab import files

print("Downloading best.pt and last.pt ...")
if os.path.exists(best_model_path):
    files.download(best_model_path)
if os.path.exists(last_model_path):
    files.download(last_model_path)
